# v2 — RoBERTa-large (is a bigger model worth it?)
Runtime → T4 GPU → Run all. ~2.5 h. See this folder's README.md.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
# Checkpoints go to Google Drive so a disconnect can be resumed. If the mount fails
# ("credential propagation was unsuccessful" happens with several Google accounts in one browser),
# fall back to the VM's disk: training still works, but a disconnect restarts from zero.
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE = '/content/drive/MyDrive/btp_v2_roberta_large'
except Exception as e:
    print(f'Drive mount failed ({e}); checkpoints stay on the VM.')
    DRIVE = '/content/btp_v2_roberta_large'
!mkdir -p {DRIVE}
!test -d /content/BTP || git clone -q https://github.com/Abhijeet-SP/BTP.git /content/BTP
%cd /content/BTP
!git fetch -q origin && git reset -q --hard origin/main && git log --oneline -1
RESULTS = 'versions/v2_roberta_large/results'
!pip -q install -U transformers datasets accelerate scikit-learn

In [ ]:
# same deterministic splits as v1
!python prepare_data.py

In [ ]:
# ~2.2 h on a T4: 355M params, batch 8 x grad-accum 4 to fit in 16GB
!python finetune_roberta.py --model roberta-large --name roberta_large_3class --lr 1e-5 --batch-size 8 --grad-accum 4 --eval-steps 1875 --results-dir {RESULTS} --ckpt-dir {DRIVE}/ckpt

In [ ]:
import os
assert os.path.exists('models/roberta_large_3class/config.json'), 'Training failed: scroll up. Re-run the cell to resume from the checkpoint.'
print('Training finished OK')

In [ ]:
import json
m = json.load(open(f'{RESULTS}/metrics.json'))
for k in ['roberta_large_3class', 'roberta_large_natural', 'roberta_large_natural_prior']:
    if k in m:
        v = m[k]
        print(f"{k:28s} acc {v['accuracy']:.4f}  macro-F1 {v['macro_f1']:.4f}  "
              f"off-by-one {v['off_by_one_accuracy']:.4f}  MAE {v['mae_classes']:.3f}  QWK {v['quadratic_weighted_kappa']:.4f}")

In [ ]:
!zip -qr btp_v2_roberta_large.zip models/roberta_large_3class versions/v2_roberta_large/results && ls -lh btp_v2_roberta_large.zip
!cp btp_v2_roberta_large.zip {DRIVE}/
from google.colab import files
files.download('btp_v2_roberta_large.zip')